In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
ground_truth[10]

{'question': 'How do I join the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [4]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [5]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [6]:
q = ground_truth[10]
q

{'question': 'How do I join the Office Hours or live workshop stream if I’m a student?',
 'document': '489dd1c9d9'}

In [7]:
doc_idx[q['document']]

{'id': '489dd1c9d9',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
 'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'}

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
    course='llm-zoomcamp',
)

In [10]:
q['question']

'How do I join the Office Hours or live workshop stream if I’m a student?'

In [11]:
answer = assistant.rag(q['question'])

In [12]:
assistant.total_cost()

0.001158

In [13]:
print(answer)

If you’re a student, you join live Office Hours or workshop sessions via **YouTube Live**. The **Zoom link is only for instructors/presenters/TAs**.

What to do:
- Watch on the **DataTalksClub YouTube channel**
- Check the **announcements channel on Telegram or Slack** for the live video URL before it starts
- Ask questions through **Slido** when the session is live
- **Don’t post questions in chat**, since they may be missed




In [14]:
doc_id = q["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'

In [15]:
rag_result = {
    "question": q['question'],
    "answer_llm": answer,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'How do I join the Office Hours or live workshop stream if I’m a student?',
 'answer_llm': 'If you’re a student, you join live Office Hours or workshop sessions via **YouTube Live**. The **Zoom link is only for instructors/presenters/TAs**.\n\nWhat to do:\n- Watch on the **DataTalksClub YouTube channel**\n- Check the **announcements channel on Telegram or Slack** for the live video URL before it starts\n- Ask questions through **Slido** when the session is live\n- **Don’t post questions in chat**, since they may be missed\n\n',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in cha

In [16]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [17]:
record = generate_rag_answer(q)
record

{'question': 'How do I join the Office Hours or live workshop stream if I’m a student?',
 'answer_llm': 'If you’re a student, join the live sessions via **YouTube Live**. The **video URL** is posted in the **announcements channel on Telegram and Slack** before the session starts, and you can also watch on the **DataTalksClub YouTube channel**.\n\nFor questions, use **Slido** during the live stream; the link is pinned in chat when live. Avoid posting questions directly in chat because they may be missed.',
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room 

In [18]:
assistant.total_cost()

0.002262

In [19]:
assistant.reset_usage()

In [20]:
assistant.total_cost()

0.0

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [22]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/515 [00:00<?, ?it/s]

In [23]:
results[:10]

[{'question': 'I just found this course late — can I still join now, or is it too late to start?',
  'answer_llm': 'Yes, you can still join now. You can start learning anytime.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'If I join after the course has already started, am I still able to get a certificate?',
  'answer_llm': 'Yes, but only if you complete the course with the live cohort and submit your project while submissions are still being accepted. You can’t get a certificate in self-paced mode.',
  'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to qualify for a certificate if I join the course 

In [24]:
df_results = pd.DataFrame(results)

In [25]:
df_results.head()

,question,answer_llm,answer_orig,document
0,I just found this course late — can I still jo...,"Yes, you can still join now. You can start lea...","Yes, but if you want to receive a certificate,...",74eb249bbf
1,If I join after the course has already started...,"Yes, but only if you complete the course with ...","Yes, but if you want to receive a certificate,...",74eb249bbf
2,What do I need to do to qualify for a certific...,To qualify for a certificate after joining lat...,"Yes, but if you want to receive a certificate,...",74eb249bbf
3,Is it okay to start the course after it begins...,"Yes, you can start whenever you want, even aft...","Yes, but if you want to receive a certificate,...",74eb249bbf
4,"When I join late, do I only need to submit the...","No. To get certified, you need to submit your ...","Yes, but if you want to receive a certificate,...",74eb249bbf


In [26]:
assistant.total_cost()

0.5214322500000005

In [27]:
df_results.to_csv("data/rag-answers-new.csv", index=False)